# 41 - Improved Experiments: Focal Loss + FER2013 Pre-Training

**Strategi baru untuk meningkatkan performa:**
1. **Focal Loss** (Lin et al., 2017) — fokus ke sampel sulit, ganti CrossEntropyLoss
2. **FER2013 Pre-Training** — ResNet18 pre-train di FER2013 (domain-specific TL)
3. **Kombinasi** — Focal Loss + FER2013 backbone

**Dataset:** Front-only 4-class (best config)
**Model:** CNN TL, Intermediate TL, Late Fusion TL
**Evaluasi:** Single Split (sama dengan eksperimen sebelumnya)

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
    EmotionCNNTransferFER, IntermediateFusionTransferFER,
)
from training.utils import (
    EmotionImageDataset, EmotionLandmarkDataset, EmotionMultimodalDataset,
    get_class_weights, train_model, full_evaluation,
    FocalLoss,
    plot_training_history, plot_confusion_matrix,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_frontonly_4class"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly" / "improved"
FER_WEIGHTS = PROJECT_ROOT / "models" / "pretrained" / "resnet18_fer2013.pth"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
NUM_CLASSES = 4
EMOTIONS = ["neutral", "happy", "sad", "negative"]

print(f"FER2013 weights: {FER_WEIGHTS.exists()}")
print("Setup complete.")

Device: cuda
GPU: Tesla T4
FER2013 weights: True
Setup complete.


## Load Data

In [2]:
from torch.utils.data import DataLoader

train_img_ds = EmotionImageDataset(DATASET_DIR / "X_train_images.npy", DATASET_DIR / "y_train.npy")
val_img_ds = EmotionImageDataset(DATASET_DIR / "X_val_images.npy", DATASET_DIR / "y_val.npy")
test_img_ds = EmotionImageDataset(DATASET_DIR / "X_test_images.npy", DATASET_DIR / "y_test.npy")

train_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_train_landmarks.npy", DATASET_DIR / "y_train.npy")
val_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_val_landmarks.npy", DATASET_DIR / "y_val.npy")
test_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_test_landmarks.npy", DATASET_DIR / "y_test.npy")

train_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_train_images.npy", DATASET_DIR / "X_train_landmarks.npy", DATASET_DIR / "y_train.npy")
val_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_val_images.npy", DATASET_DIR / "X_val_landmarks.npy", DATASET_DIR / "y_val.npy")
test_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_test_images.npy", DATASET_DIR / "X_test_landmarks.npy", DATASET_DIR / "y_test.npy")

train_img = DataLoader(train_img_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_img = DataLoader(val_img_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_img = DataLoader(test_img_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_lm = DataLoader(train_lm_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_lm = DataLoader(val_lm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_lm = DataLoader(test_lm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_mm = DataLoader(train_mm_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_mm = DataLoader(val_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_mm = DataLoader(test_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

weights = get_class_weights(DATASET_DIR, device)
print(f"Train: {len(train_img_ds)} | Val: {len(val_img_ds)} | Test: {len(test_img_ds)}")
print(f"Class weights: {weights}")

Train: 5348 | Val: 707 | Test: 1036
Class weights: None


## Helper

In [3]:
all_results = {}

def run_experiment(name, model, criterion, train_loader, val_loader, test_loader,
                  model_type, lr=0.00005):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    save_path = str(OUTPUT_DIR / f"{name}.pth")
    
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    
    history, best_epoch = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                                       device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, EMOTIONS)
    
    print(f"  Acc={r['test_accuracy']:.4f} Macro-F1={r['test_macro_f1']:.4f}")
    all_results[name] = {
        "accuracy": float(r["test_accuracy"]),
        "macro_f1": float(r["test_macro_f1"]),
        "weighted_f1": float(r["test_weighted_f1"]),
    }
    return r

## Experiment 1: Focal Loss (ImageNet TL)

In [4]:
# CNN TL + Focal Loss
model = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("CNN_TL_FocalLoss", model, criterion, train_img, val_img, test_img, "cnn")

# FCNN + Focal Loss
model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("FCNN_FocalLoss", model, criterion, train_lm, val_lm, test_lm, "fcnn", lr=0.0001)

# Intermediate TL + Focal Loss
model = IntermediateFusionTransfer(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("IntermediateTL_FocalLoss", model, criterion, train_mm, val_mm, test_mm, "fusion")


  CNN_TL_FocalLoss
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      0.4082     0.7147     0.2245    0.9321   0.3623   0.000050  (20.9s)


     2      0.2165     0.8485     0.1300    0.9392   0.3557   0.000050  (20.3s)


     3      0.1648     0.8706     0.1409    0.9222   0.3276   0.000050  (20.2s)


     4      0.1261     0.8979     0.1333    0.9066   0.3503   0.000050  (20.1s)


...

    23      0.0129     0.9907     0.1623    0.8911   0.3639   0.000025  (19.2s)


    24      0.0098     0.9929     0.1756    0.8854   0.3326   0.000025  (19.3s)


    25      0.0120     0.9910     0.1991    0.8812   0.3280   0.000025  (19.3s)

Early stopping at epoch 25. Best epoch: 10 (val_f1=0.4716)

Best: epoch 10, val_acc=0.9335, val_f1=0.4716
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly/improved/IntermediateTL_FocalLoss.pth


Test Loss: 0.1154
Test Accuracy: 0.9392
Test Macro F1: 0.3502
Test Weighted F1: 0.9333

Classification Report:
              precision    recall  f1-score   support

     neutral       0.97      0.98      0.98       981
       happy       0.00      0.00      0.00        10
         sad       0.25      0.10      0.15        29
    negative       0.25      0.31      0.28        16

    accuracy                           0.94      1036
   macro avg       0.37      0.35      0.35      1036
weighted avg       0.93      0.94      0.93      1036

  Acc=0.9392 Macro-F1=0.3502


{'test_loss': 0.1153728700886592,
 'test_accuracy': 0.9391891891891891,
 'test_macro_f1': 0.3502099722414228,
 'test_weighted_f1': 0.9332541528465791,
 'predictions': array([0, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'labels': array([1, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'probabilities': array([[0.8473621 , 0.04085113, 0.06244296, 0.04934375],
        [0.8457142 , 0.05348377, 0.04527751, 0.05552449],
        [0.85702354, 0.05106625, 0.04349416, 0.0484161 ],
        ...,
        [0.80369204, 0.0470176 , 0.04571329, 0.10357703],
        [0.6863853 , 0.08704033, 0.0846125 , 0.14196184],
        [0.3489745 , 0.3210591 , 0.15178159, 0.1781848 ]],
       shape=(1036, 4), dtype=float32),
 'confusion_matrix': array([[965,   5,   7,   4],
        [ 10,   0,   0,   0],
        [ 14,   1,   3,  11],
        [  6,   3,   2,   5]])}

## Experiment 2: FER2013 Pre-Training (CrossEntropyLoss)

In [5]:
# CNN FER2013 TL + CrossEntropy
model = EmotionCNNTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
run_experiment("CNN_FER2013_CE", model, criterion, train_img, val_img, test_img, "cnn")

# Intermediate FER2013 TL + CrossEntropy
model = IntermediateFusionTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
run_experiment("IntermediateTL_FER2013_CE", model, criterion, train_mm, val_mm, test_mm, "fusion")

  Loaded FER2013 pre-trained weights from /home/bs000716/MOTHER-TANK/TRAIN/models/pretrained/resnet18_fer2013.pth

  CNN_FER2013_CE
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      0.9706     0.6632     0.5389    0.9151   0.4147   0.000050  (19.4s)


     2      0.5407     0.8588     0.3695    0.9349   0.4250   0.000050  (19.1s)


     3      0.3709     0.8949     0.3605    0.9279   0.3946   0.000050  (19.1s)


     4      0.2779     0.9207     0.2640    0.9349   0.3713   0.000050  (19.0s)


...

    33      0.0156     0.9976     0.2963    0.9406   0.4291   0.000025  (19.1s)


    34      0.0149     0.9976     0.2945    0.9335   0.4563   0.000025  (19.2s)


    35      0.0131     0.9978     0.2891    0.9364   0.4820   0.000025  (19.3s)

Early stopping at epoch 35. Best epoch: 20 (val_f1=0.5501)

Best: epoch 20, val_acc=0.9392, val_f1=0.5501
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly/improved/IntermediateTL_FER2013_CE.pth


Test Loss: 0.2798
Test Accuracy: 0.9353
Test Macro F1: 0.4062
Test Weighted F1: 0.9284

Classification Report:
              precision    recall  f1-score   support

     neutral       0.96      0.98      0.97       981
       happy       0.19      0.40      0.26        10
         sad       0.43      0.10      0.17        29
    negative       0.30      0.19      0.23        16

    accuracy                           0.94      1036
   macro avg       0.47      0.42      0.41      1036
weighted avg       0.93      0.94      0.93      1036

  Acc=0.9353 Macro-F1=0.4062


{'test_loss': 0.27976655403446965,
 'test_accuracy': 0.9353281853281853,
 'test_macro_f1': 0.406169191314426,
 'test_weighted_f1': 0.928444292661721,
 'predictions': array([0, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'labels': array([1, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'probabilities': array([[9.9542159e-01, 1.9091708e-03, 1.0210638e-03, 1.6480902e-03],
        [9.9585283e-01, 1.4655365e-03, 9.3895517e-04, 1.7427437e-03],
        [9.9706858e-01, 8.4135478e-04, 6.1061484e-04, 1.4794971e-03],
        ...,
        [9.9185771e-01, 1.7190562e-03, 3.2710803e-03, 3.1521125e-03],
        [9.9045640e-01, 2.2900775e-03, 2.5880805e-03, 4.6654027e-03],
        [9.9012321e-01, 2.6429233e-03, 2.4833600e-03, 4.7505200e-03]],
       shape=(1036, 4), dtype=float32),
 'confusion_matrix': array([[959,  15,   3,   4],
        [  5,   4,   1,   0],
        [ 22,   1,   3,   3],
        [ 12,   1,   0,   3]])}

## Experiment 3: FER2013 + Focal Loss (Best Combo)

In [6]:
# CNN FER2013 TL + Focal Loss
model = EmotionCNNTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("CNN_FER2013_Focal", model, criterion, train_img, val_img, test_img, "cnn")

# Intermediate FER2013 TL + Focal Loss
model = IntermediateFusionTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("IntermediateTL_FER2013_Focal", model, criterion, train_mm, val_mm, test_mm, "fusion")

  Loaded FER2013 pre-trained weights from /home/bs000716/MOTHER-TANK/TRAIN/models/pretrained/resnet18_fer2013.pth

  CNN_FER2013_Focal
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      0.4257     0.6885     0.1527    0.9293   0.3715   0.000050  (19.1s)


     2      0.2318     0.8351     0.1061    0.9434   0.4085   0.000050  (19.2s)


     3      0.1686     0.8745     0.1069    0.9335   0.3805   0.000050  (19.1s)


     4      0.1330     0.8958     0.1021    0.9420   0.4024   0.000050  (19.0s)


...

    26      0.0063     0.9959     0.1787    0.9165   0.3999   0.000025  (19.3s)


    27      0.0098     0.9918     0.1541    0.9378   0.4104   0.000025  (19.3s)


    28      0.0055     0.9972     0.1663    0.9194   0.3970   0.000025  (19.3s)

Early stopping at epoch 28. Best epoch: 13 (val_f1=0.4788)

Best: epoch 13, val_acc=0.9335, val_f1=0.4788
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly/improved/IntermediateTL_FER2013_Focal.pth


Test Loss: 0.1565
Test Accuracy: 0.9199
Test Macro F1: 0.3434
Test Weighted F1: 0.9221

Classification Report:
              precision    recall  f1-score   support

     neutral       0.97      0.96      0.97       981
       happy       0.12      0.40      0.18        10
         sad       0.25      0.21      0.23        29
    negative       0.00      0.00      0.00        16

    accuracy                           0.92      1036
   macro avg       0.33      0.39      0.34      1036
weighted avg       0.93      0.92      0.92      1036

  Acc=0.9199 Macro-F1=0.3434


{'test_loss': 0.15647605261459774,
 'test_accuracy': 0.9198841698841699,
 'test_macro_f1': 0.34335821668530575,
 'test_weighted_f1': 0.9220511755949464,
 'predictions': array([0, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'labels': array([1, 0, 0, ..., 0, 0, 0], shape=(1036,)),
 'probabilities': array([[0.88737875, 0.04797735, 0.02448996, 0.04015386],
        [0.91012186, 0.02781578, 0.02765644, 0.03440586],
        [0.9071591 , 0.02332406, 0.03618049, 0.03333638],
        ...,
        [0.5345159 , 0.15657902, 0.07520783, 0.23369728],
        [0.7238296 , 0.13444301, 0.0255472 , 0.11618016],
        [0.8171433 , 0.10490256, 0.02413284, 0.05382129]],
       shape=(1036, 4), dtype=float32),
 'confusion_matrix': array([[943,  23,  12,   3],
        [  6,   4,   0,   0],
        [ 17,   4,   6,   2],
        [  7,   3,   6,   0]])}

## Late Fusion Variants

In [7]:
# Late Fusion: CNN_FER2013_Focal + FCNN_FocalLoss
print("\nLate Fusion: FER2013 Focal + FCNN Focal")
cnn_model = EmotionCNNTransferFER(num_classes=NUM_CLASSES).to(device)
cnn_model.load_state_dict(torch.load(OUTPUT_DIR / "CNN_FER2013_Focal.pth", map_location=device, weights_only=True))
fcnn_model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
fcnn_model.load_state_dict(torch.load(OUTPUT_DIR / "FCNN_FocalLoss.pth", map_location=device, weights_only=True))
cnn_model.eval(); fcnn_model.eval()

cnn_probs_all, fcnn_probs_all, labels_all = [], [], []
with torch.no_grad():
    for images, landmarks, labels in test_mm:
        cnn_probs_all.append(torch.softmax(cnn_model(images.to(device)), dim=1).cpu().numpy())
        fcnn_probs_all.append(torch.softmax(fcnn_model(landmarks.to(device)), dim=1).cpu().numpy())
        labels_all.append(labels.numpy())

cnn_probs = np.concatenate(cnn_probs_all)
fcnn_probs = np.concatenate(fcnn_probs_all)
lbls = np.concatenate(labels_all)

best_f1, best_w = 0, 0.5
for w in np.arange(0.0, 1.05, 0.05):
    preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
    f1 = f1_score(lbls, preds, average="macro", zero_division=0)
    if f1 > best_f1: best_f1 = f1; best_w = w; best_preds = preds

acc = accuracy_score(lbls, best_preds)
wf1 = f1_score(lbls, best_preds, average="weighted", zero_division=0)
print(f"  LateFusion_FER2013_Focal: w={best_w:.2f} Acc={acc:.4f} F1={best_f1:.4f}")
all_results["LateFusion_FER2013_Focal"] = {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w}


Late Fusion: FER2013 Focal + FCNN Focal


  LateFusion_FER2013_Focal: w=1.00 Acc=0.9344 F1=0.4011


## Ringkasan & Perbandingan

In [8]:
print("="*75)
print("RINGKASAN IMPROVED EXPERIMENTS (4-class Front-Only)")
print("="*75)
print(f"{'Experiment':<35} {'Macro F1':>10} {'Accuracy':>10}")
print("-"*60)
for k in sorted(all_results.keys(), key=lambda x: -all_results[x]["macro_f1"]):
    r = all_results[k]
    print(f"{k:<35} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f}")

print("\nPerbandingan dengan baseline (sebelumnya):")
print(f"  Intermediate TL B1 (CE):       0.412")
print(f"  Late Fusion B3 (CE):           0.394")
print(f"  FCNN B3 (CE):                  0.361")

# Save
with open(OUTPUT_DIR / "improved_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR / 'improved_results.json'}")

RINGKASAN IMPROVED EXPERIMENTS (4-class Front-Only)
Experiment                            Macro F1   Accuracy
------------------------------------------------------------
IntermediateTL_FER2013_CE               0.4062     0.9353
CNN_FER2013_Focal                       0.4011     0.9344
LateFusion_FER2013_Focal                0.4011     0.9344
CNN_TL_FocalLoss                        0.3543     0.9431
IntermediateTL_FocalLoss                0.3502     0.9392
CNN_FER2013_CE                          0.3497     0.9411
IntermediateTL_FER2013_Focal            0.3434     0.9199
FCNN_FocalLoss                          0.2528     0.8707

Perbandingan dengan baseline (sebelumnya):
  Intermediate TL B1 (CE):       0.412
  Late Fusion B3 (CE):           0.394
  FCNN B3 (CE):                  0.361

Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly/improved/improved_results.json
